# Importing libraries

In [4]:
# Basic libraries
import pandas as pd
from datasets import load_dataset
import time
import pickle


# Classification models
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import MultinomialNB

# Vectorizers
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Utilities and metrics
from itertools import product
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from memory_profiler import memory_usage

# Preprocessing
import nltk
import re

# Download nltk resources
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Rafael\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Setting seeds

In [5]:
s1 = 2
s2 = 3
s3 = 5

seeds = [s1, s2, s3]

# Importing datasets

In [6]:
ds = load_dataset("dair-ai/emotion", "split")

train = ds['train'].to_pandas()
val = ds['validation'].to_pandas()
test = ds['test'].to_pandas()

train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    16000 non-null  object
 1   label   16000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 250.1+ KB


# Dataset preprocessing

In [7]:
stop_words = set(nltk.corpus.stopwords.words('english'))
lemmatizer = nltk.stem.WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()

    text = re.sub(r'[^\w\s]', '', text)
    
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]

    return ' '.join(words)
train['text'] = train['text'].apply(preprocess_text)

# GridSearch implementation

In [8]:
vectorizers = [
    TfidfVectorizer(),
    CountVectorizer()
]

models = {
    'RandomForest': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [50, 100, 150],
            'criterion': ['gini', 'entropy', 'log_loss']
        }
    },
    'SVC': {
        'model': SVC(),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['linear', 'rbf', 'sigmoid']
        }
    },
    'MultinomialNB': {
        'model': MultinomialNB(),
        'params': {
            'alpha': [0.01, 0.1, 1.0]
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(max_iter=1000),
        'params': {
            'C': [0.1, 1, 10],
            'penalty': ['l2']
        }
    },
    'KNeighbors': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7],
            'algorithm': ['ball_tree', 'kd_tree', 'brute']
        }
    },
    'DecisionTree': {
        'model': DecisionTreeClassifier(),
        'params': {
            'criterion': ['gini', 'entropy', 'log_loss'],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(),
        'params': {
            'n_estimators': [100, 150, 200],
            'criterion': ['friedman_mse', 'squared_error'],
        }
    },
    'AdaBoost': {
        'model': AdaBoostClassifier(),
        'params': {
            'n_estimators': [50, 100, 150],
            'learning_rate': [0.01, 0.1, 1.0]
        }
    },
    'SGD': {
        'model': SGDClassifier(),
        'params': {
            'alpha': [0.0001, 0.001, 0.01],
            'penalty': ['l2', 'l1', 'elasticnet']
        }
    }
}

In [9]:
columns = ['seed', 'vectorizer', 'model', 'params', 'accuracy', 'training_time', 'prediction_time', 'peak_memory_train', 'peak_memory_prediction']

classes = sorted(train['label'].unique())
for c in classes:
    columns.extend([
        f'precision_class_{c}',
        f'recall_class_{c}',
        f'f1_class_{c}'
    ])

results = pd.DataFrame(columns=columns)

In [10]:
for seed in seeds:
    print(f"Processing seed: {seed}")
    for vectorizer in vectorizers:
        print(f"Processing vectorizer: {vectorizer.__class__.__name__}")
        for name, info in models.items():
            print(f"Processing model: {name}")

            model = info['model']
            param_grid = info['params']
            param_combinations = product(*param_grid.values())
            
            for combination in param_combinations:
                params = dict(zip(param_grid.keys(), combination))
                model.set_params(**params)
                if 'random_state' in model.get_params():
                    model.set_params(random_state=seed)

                print(f"Training {name} with params {params} and vectorizer {vectorizer.__class__.__name__}")

                pipeline = Pipeline([
                    ('vectorizer', vectorizer),
                    ('model', model)
                ])

                def train_model():
                    pipeline.fit(train['text'], train['label'])

                def predict_model():
                    return pipeline.predict(val['text'])

                # Training Phase
                start_time = time.perf_counter()
                peak_memory_train = memory_usage(train_model, max_usage=True)
                train_time = time.perf_counter() - start_time
                print(f"Training time: {train_time}")
                print(f"Peak memory usage during training: {peak_memory_train} MB")

                # Prediction Phase
                start_time = time.perf_counter()
                peak_memory_pred, y_pred = memory_usage(predict_model, max_usage=True, retval=True)
                prediction_time = time.perf_counter() - start_time
                print(f"Prediction time: {prediction_time}")
                print(f"Peak memory usage during prediction: {peak_memory_pred} MB")
                
                accuracy = accuracy_score(val['label'], y_pred)
                precisions, recalls, f1s, supports = precision_recall_fscore_support(val['label'], y_pred, average=None, labels=classes, zero_division=0)

                result_dict = {
                    'seed': seed,
                    'vectorizer': vectorizer.__class__.__name__,
                    'model': name,
                    'params': params,
                    'accuracy': accuracy,
                    'training_time': train_time,
                    'prediction_time': prediction_time,
                    'peak_memory_train': peak_memory_train,
                    'peak_memory_prediction': peak_memory_pred,
                }

                for i, c in enumerate(classes):
                    result_dict[f'precision_class_{c}'] = precisions[i]
                    result_dict[f'recall_class_{c}'] = recalls[i]
                    result_dict[f'f1_class_{c}'] = f1s[i]

                result = pd.DataFrame([result_dict])
                results = pd.concat([results, result], ignore_index=True)
                print("-"*100)

results.to_csv('results/results_sklearn_multiclass3.csv', index=False)

Processing seed: 2
Processing vectorizer: TfidfVectorizer
Processing model: RandomForest
Training RandomForest with params {'n_estimators': 50, 'criterion': 'gini'} and vectorizer TfidfVectorizer
Training time: 8.392934100000275
Peak memory usage during training: 423.02734375 MB
Prediction time: 1.0655908999997337
Peak memory usage during prediction: 415.86328125 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'entropy'} and vectorizer TfidfVectorizer


C:\Users\Rafael\AppData\Local\Temp\ipykernel_8692\4276233647.py:66: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, result], ignore_index=True)


Training time: 9.704741199999717
Peak memory usage during training: 425.37109375 MB
Prediction time: 1.0338348000000224
Peak memory usage during prediction: 425.49609375 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'log_loss'} and vectorizer TfidfVectorizer
Training time: 9.689865800000007
Peak memory usage during training: 425.5078125 MB
Prediction time: 1.0339995999997882
Peak memory usage during prediction: 425.6484375 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 100, 'criterion': 'gini'} and vectorizer TfidfVectorizer
Training time: 15.988385800000287
Peak memory usage during training: 475.71875 MB
Prediction time: 1.1388790000000881
Peak memory usage during prediction: 469.41015625 MB
----------------------------------------------------------------

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\neighbors\_base.py:583: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\neighbors\_base.py:583: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")


Training time: 1.113829700000224
Peak memory usage during training: 552.30078125 MB
Prediction time: 1.038736500000141
Peak memory usage during prediction: 1009.9921875 MB
----------------------------------------------------------------------------------------------------
Training KNeighbors with params {'n_neighbors': 3, 'algorithm': 'kd_tree'} and vectorizer TfidfVectorizer


...

Training time: 2.7897093000001405
Peak memory usage during training: 562.296875 MB
Prediction time: 1.0139784000002692
Peak memory usage during prediction: 559.16015625 MB
----------------------------------------------------------------------------------------------------
Training AdaBoost with params {'n_estimators': 150, 'learning_rate': 0.1} and vectorizer CountVectorizer


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Training time: 2.690187100000003
Peak memory usage during training: 561.9609375 MB
Prediction time: 1.011452200000349
Peak memory usage during prediction: 559.33984375 MB
----------------------------------------------------------------------------------------------------
Training AdaBoost with params {'n_estimators': 150, 'learning_rate': 1.0} and vectorizer CountVectorizer


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Training time: 2.7604157000005216
Peak memory usage during training: 561.9921875 MB
Prediction time: 1.0172093000001041
Peak memory usage during prediction: 558.92578125 MB
----------------------------------------------------------------------------------------------------
Processing model: SGD
Training SGD with params {'alpha': 0.0001, 'penalty': 'l2'} and vectorizer CountVectorizer
Training time: 1.2712901999993846
Peak memory usage during training: 562.27734375 MB
Prediction time: 1.8133518000004187
Peak memory usage during prediction: 553.91796875 MB
----------------------------------------------------------------------------------------------------
Training SGD with params {'alpha': 0.0001, 'penalty': 'l1'} and vectorizer CountVectorizer
Training time: 0.7642344000005323
Peak memory usage during training: 562.11328125 MB
Prediction time: 1.7975145000000339
Peak memory usage during prediction: 553.6640625 MB
--------------------------------------------------------------------------

# Process results

In [11]:
results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   seed                    396 non-null    object 
 1   vectorizer              396 non-null    object 
 2   model                   396 non-null    object 
 3   params                  396 non-null    object 
 4   accuracy                396 non-null    float64
 5   training_time           396 non-null    float64
 6   prediction_time         396 non-null    float64
 7   peak_memory_train       396 non-null    float64
 8   peak_memory_prediction  396 non-null    float64
 9   precision_class_0       396 non-null    float64
 10  recall_class_0          396 non-null    float64
 11  f1_class_0              396 non-null    float64
 12  precision_class_1       396 non-null    float64
 13  recall_class_1          396 non-null    float64
 14  f1_class_1              396 non-null    fl

In [12]:
results.head()

,seed,vectorizer,model,params,accuracy,training_time,prediction_time,peak_memory_train,peak_memory_prediction,precision_class_0,...,f1_class_2,precision_class_3,recall_class_3,f1_class_3,precision_class_4,recall_class_4,f1_class_4,precision_class_5,recall_class_5,f1_class_5
0,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'gini'}",0.890,8.392934,1.065591,423.027344,415.863281,0.908273,...,0.817647,0.896947,0.854545,0.875233,0.839450,0.863208,0.851163,0.911765,0.765432,0.832215
1,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'entropy'}",0.884,9.704741,1.033835,425.371094,425.496094,0.911602,...,0.814815,0.894737,0.865455,0.879852,0.823529,0.858491,0.840647,0.887324,0.777778,0.828947
2,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'log_loss'}",0.884,9.689866,1.034000,425.507812,425.648438,0.911602,...,0.814815,0.894737,0.865455,0.879852,0.823529,0.858491,0.840647,0.887324,0.777778,0.828947
3,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'gini'}",0.890,15.988386,1.138879,475.718750,469.410156,0.909747,...,0.810496,0.904580,0.861818,0.882682,0.836364,0.867925,0.851852,0.897059,0.753086,0.818792
4,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'entropy'}",0.886,18.734822,1.136784,483.570312,476.863281,0.918819,...,0.812680,0.890977,0.861818,0.876155,0.814159,0.867925,0.840183,0.910448,0.753086,0.824324


In [13]:
results['params'] = results['params'].astype(str)
results_avg_seed = results.groupby(['model', 'vectorizer', 'params']).mean().reset_index()
results_avg_seed['f1_avg'] = results_avg_seed[[col for col in results_avg_seed.columns if 'f1_class' in col]].mean(axis=1)
results_avg_seed

,model,vectorizer,params,seed,accuracy,training_time,prediction_time,peak_memory_train,peak_memory_prediction,precision_class_0,...,precision_class_3,recall_class_3,f1_class_3,precision_class_4,recall_class_4,f1_class_4,precision_class_5,recall_class_5,f1_class_5,f1_avg
0,AdaBoost,CountVectorizer,"{'n_estimators': 100, 'learning_rate': 0.01}",3.333333,0.3515,2.161124,1.074813,562.247396,554.589844,0.000000,...,0.000000,0.000000,0.000000,1.000000,0.004717,0.009390,0.000000,0.000000,0.000000,0.089074
1,AdaBoost,CountVectorizer,"{'n_estimators': 100, 'learning_rate': 0.1}",3.333333,0.3600,2.058383,1.022473,562.220052,554.221354,0.000000,...,0.000000,0.000000,0.000000,0.439024,0.084906,0.142292,0.500000,0.012346,0.024096,0.116544
2,AdaBoost,CountVectorizer,"{'n_estimators': 100, 'learning_rate': 1.0}",3.333333,0.3685,2.047303,1.013030,562.179688,554.363281,0.000000,...,0.285714,0.007273,0.014184,0.483333,0.136792,0.213235,0.800000,0.197531,0.316832,0.220409
3,AdaBoost,CountVectorizer,"{'n_estimators': 150, 'learning_rate': 0.01}",3.333333,0.3515,2.833372,1.030625,560.151042,556.955729,0.000000,...,0.000000,0.000000,0.000000,1.000000,0.004717,0.009390,0.000000,0.000000,0.000000,0.089074
4,AdaBoost,CountVectorizer,"{'n_estimators': 150, 'learning_rate': 0.1}",3.333333,0.3600,2.741074,1.027341,559.048177,556.981771,0.000000,...,0.000000,0.000000,0.000000,0.425000,0.080189,0.134921,0.500000,0.024691,0.047059,0.119177
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,SVC,TfidfVectorizer,"{'C': 1, 'kernel': 'rbf'}",3.333333,0.8605,31.526554,2.100625,789.079427,565.692708,0.865772,...,0.915612,0.789091,0.847656,0.855615,0.754717,0.802005,0.944444,0.629630,0.755556,0.816410
128,SVC,TfidfVectorizer,"{'C': 1, 'kernel': 'sigmoid'}",3.333333,0.8895,14.556889,1.533473,729.212240,565.088542,0.895470,...,0.906716,0.883636,0.895028,0.815166,0.811321,0.813239,0.907692,0.728395,0.808219,0.857992
129,SVC,TfidfVectorizer,"{'C': 10, 'kernel': 'linear'}",3.333333,0.8710,12.516618,1.472892,705.811198,566.057292,0.871528,...,0.897727,0.861818,0.879406,0.831683,0.792453,0.811594,0.833333,0.802469,0.817610,0.843550
130,SVC,TfidfVectorizer,"{'C': 10, 'kernel': 'rbf'}",3.333333,0.8690,33.440879,2.098567,787.036458,566.333333,0.872881,...,0.917355,0.807273,0.858801,0.859296,0.806604,0.832117,0.894737,0.629630,0.739130,0.825004


In [14]:
best_result = results_avg_seed.loc[results_avg_seed['f1_avg'].idxmax()]

best_model = models[best_result['model']]['model']
best_params = eval(best_result['params'])
best_model.set_params(**best_params)

best_result_vectorizer = eval(best_result['vectorizer'])()

pipeline = Pipeline([
    ('vectorizer', best_result_vectorizer),
    ('model', best_model)
])

pipeline.fit(train['text'], train['label'])
y_pred = pipeline.predict(test['text'])

accuracy = accuracy_score(test['label'], y_pred)
precisions, recalls, f1s, supports = precision_recall_fscore_support(test['label'], y_pred, average=None, labels=classes, zero_division=0)

print(f"Best model: {best_result['model']}")
print(f"Best model params: {best_result['params']}")
print(f"Best vectorizer: {best_result['vectorizer']}")
print(f"Best accuracy: {accuracy}\n")

for i, c in enumerate(classes):
    print(f"Class {c}")
    print(f"Precision: {precisions[i]}")
    print(f"Recall: {recalls[i]}")
    print(f"F1: {f1s[i]}")
    print(f"Support: {supports[i]}\n")

Best model: SGD
Best model params: {'alpha': 0.0001, 'penalty': 'l1'}
Best vectorizer: TfidfVectorizer
Best accuracy: 0.9005

Class 0
Precision: 0.9426086956521739
Recall: 0.9328743545611016
F1: 0.9377162629757786
Support: 581

Class 1
Precision: 0.8955223880597015
Recall: 0.9496402877697842
F1: 0.9217877094972067
Support: 695

Class 2
Precision: 0.8394160583941606
Recall: 0.7232704402515723
F1: 0.777027027027027
Support: 159

Class 3
Precision: 0.8920863309352518
Recall: 0.9018181818181819
F1: 0.8969258589511754
Support: 275

Class 4
Precision: 0.8766519823788547
Recall: 0.8883928571428571
F1: 0.8824833702882483
Support: 224

Class 5
Precision: 0.8043478260869565
Recall: 0.5606060606060606
F1: 0.6607142857142857
Support: 66



In [15]:
with open('models/best_model_sklearn_multiclass3.pkl', 'wb') as f:
    pickle.dump(pipeline, f)